# Parking con DDPG+HER

In [1]:
import torch
import gymnasium as gym
import stable_baselines3
from stable_baselines3 import DDPG, HerReplayBuffer
from stable_baselines3.common.noise import NormalActionNoise

import numpy as np
import time

2.5.1+cu121
1.0.0
2.5.0


## Entendiendo el ambiente

In [3]:
import highway_env
env = gym.make('parking-v0', render_mode = 'human')

episode = 2
for episode in range(1, episode+1):
    state, _ = env.reset()
    done = False
    episode_reward = 0

    while not done:
        action = env.action_space.sample()
        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        episode_reward += reward
    print(f'Episodio: {episode}, Retorno: {episode_reward}')
env.close()

Episode: 1, Reward: -16.676295810605076
Episode: 2, Reward: -48.22685355752447


In [4]:
print(env.observation_space)

Dict('achieved_goal': Box(-inf, inf, (6,), float64), 'desired_goal': Box(-inf, inf, (6,), float64), 'observation': Box(-inf, inf, (6,), float64))


In [5]:
print(env.observation_space['achieved_goal'])

Box(-inf, inf, (6,), float64)


In [6]:
print(env.observation_space['desired_goal'])

Box(-inf, inf, (6,), float64)


In [7]:
print(env.observation_space['observation'])

Box(-inf, inf, (6,), float64)


In [8]:
print(env.reset())

(OrderedDict([('observation', array([ 0.        ,  0.        , -0.        ,  0.        , -0.94341708,
        0.33160853])), ('achieved_goal', array([ 0.        ,  0.        , -0.        ,  0.        , -0.94341708,
        0.33160853])), ('desired_goal', array([-2.200000e-01, -1.400000e-01,  0.000000e+00,  0.000000e+00,
        6.123234e-17, -1.000000e+00]))]), {'speed': 0, 'crashed': False, 'action': array([-0.08194144, -0.688533  ], dtype=float32), 'is_success': False})


In [9]:
print(env.action_space)
print(env.action_space.sample())

Box(-1.0, 1.0, (2,), float32)
[ 0.60979474 -0.9373003 ]


## Agente

In [11]:
env = gym.make('parking-v0')

#Ruido para exploración en DDPG
num_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(num_actions), sigma=0.1 * np.ones(num_actions))

#Agente
model = DDPG("MultiInputPolicy", #MultiInputPolicy cuando se usa dict obs space
             env,
             learning_rate=0.001,
             buffer_size=1000000,
             batch_size=256,
             gamma=0.95,
             action_noise=action_noise,
             replay_buffer_class=HerReplayBuffer,
             policy_kwargs=dict(net_arch=[256, 256, 256]),
             verbose = 0)

model.learn(total_timesteps=50000, progress_bar = True)
model.save("parking_ddpg_her")

Output()

## Testing

In [18]:
env = gym.make('parking-v0', render_mode = 'human')
model = DDPG.load("parking_ddpg_her", env=env)

state, _ = env.reset()
episode_reward = 0
for _ in range(100):
    action, _ = model.predict(state, deterministic=True)
    state, reward, done, truncated, info = env.step(action)
    episode_reward += reward
    if done or info.get("is_success"):
        print("Retorno:", episode_reward, "Exito?", info.get("is_success"))
        episode_reward = 0.0
        state,_ = env.reset()
env.close()

Retorno: -14.847761785814338 Exito? False
Retorno: -7.558358574473761 Exito? True
Retorno: -7.376489813241571 Exito? True
Retorno: -9.567946220834031 Exito? True
